# Motor Imagery 4-Class FBCSP Tutorial

This notebook demonstrates how to use the Motor Imagery FBCSP (Filter Bank Common Spatial Patterns) library for classifying four-class motor imagery tasks.

## Overview

Motor imagery classification is a brain-computer interface (BCI) task where we classify EEG signals based on imagined movements:
- **Class 0**: Left Hand movement
- **Class 1**: Right Hand movement
- **Class 2**: Feet movement
- **Class 3**: Tongue movement

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add src to path
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from classifier import MotorImageryClassifier
from preprocessing import Preprocessor
from data_loader import DataLoader
from config import Config

print("Libraries imported successfully!")

## 1. Generate Synthetic Data

For demonstration, we'll create synthetic motor imagery data with distinct frequency characteristics for each class.

In [ ]:
# Generate synthetic data
n_trials = 200
n_channels = 8
n_samples = int(4.0 * Config.SAMPLE_RATE)  # 4 seconds at 250 Hz

X, y = DataLoader.create_synthetic_data(
    n_trials=n_trials,
    n_channels=n_channels,
    n_samples=n_samples,
    n_classes=4,
    sample_rate=Config.SAMPLE_RATE
)

print(f"Data shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Classes: {np.unique(y)}")
print(f"Class distribution: {np.bincount(y)}")

## 2. Visualize Sample EEG Data

In [ ]:
# Plot sample trials from each class
fig, axes = plt.subplots(4, 1, figsize=(12, 10))

for class_idx in range(4):
    # Get first trial of this class
    trial_idx = np.where(y == class_idx)[0][0]
    
    # Plot first channel
    time = np.arange(n_samples) / Config.SAMPLE_RATE
    axes[class_idx].plot(time, X[trial_idx, 0, :])
    axes[class_idx].set_title(f"Class {class_idx}: {Config.CLASS_LABELS[class_idx]}")
    axes[class_idx].set_xlabel("Time (s)")
    axes[class_idx].set_ylabel("Amplitude")
    axes[class_idx].grid(True)

plt.tight_layout()
plt.show()

## 3. Preprocess the Data

In [ ]:
# Create preprocessor
preprocessor = Preprocessor(
    sample_rate=Config.SAMPLE_RATE,
    low_freq=Config.BANDPASS_LOW,
    high_freq=Config.BANDPASS_HIGH,
    filter_order=Config.FILTER_ORDER
)

# Preprocess data
X_preprocessed = preprocessor.preprocess(X)

print("Preprocessing complete!")
print(f"Preprocessed data shape: {X_preprocessed.shape}")

## 4. Split Data into Train and Test Sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_preprocessed, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} trials")
print(f"Test set: {X_test.shape[0]} trials")

## 5. Train the FBCSP Classifier

In [ ]:
# Create classifier
classifier = MotorImageryClassifier(
    classifier_type=Config.CLASSIFIER_TYPE,
    filter_bank=Config.FILTER_BANK,
    n_components=Config.N_CSP_COMPONENTS,
    sample_rate=Config.SAMPLE_RATE
)

# Train
print("Training classifier...")
classifier.fit(X_train, y_train)
print("Training complete!")

## 6. Evaluate Performance

In [ ]:
# Test accuracy
test_accuracy = classifier.score(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.2%}")

# Cross-validation
print("\nPerforming cross-validation...")
cv_results = classifier.cross_validate(X_preprocessed, y, cv=5)
print(f"Mean CV Accuracy: {cv_results['mean_accuracy']:.2%} ± {cv_results['std_accuracy']:.2%}")

## 7. Visualize Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Make predictions
predictions = classifier.predict(X_test)

# Create confusion matrix
cm = confusion_matrix(y_test, predictions)
class_names = [Config.CLASS_LABELS[i] for i in range(4)]

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 8. Visualize Prediction Probabilities

In [ ]:
# Get probabilities for a few test samples
sample_indices = range(min(10, len(X_test)))
probabilities = classifier.predict_proba(X_test[sample_indices])

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(sample_indices))
width = 0.2

for class_idx in range(4):
    offset = width * (class_idx - 1.5)
    ax.bar(x + offset, probabilities[:, class_idx], width, 
           label=Config.CLASS_LABELS[class_idx])

ax.set_xlabel('Sample Index')
ax.set_ylabel('Probability')
ax.set_title('Prediction Probabilities for Test Samples')
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Save the Trained Model

In [ ]:
import pickle

model_data = {
    'classifier': classifier,
    'preprocessor': preprocessor,
    'config': {
        'sample_rate': Config.SAMPLE_RATE,
        'n_channels': Config.N_CHANNELS,
        'class_labels': Config.CLASS_LABELS
    }
}

with open('motor_imagery_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("Model saved to 'motor_imagery_model.pkl'")

## Conclusion

This tutorial demonstrated:
1. Loading/generating motor imagery EEG data
2. Preprocessing the data (filtering, normalization)
3. Training a FBCSP classifier for 4-class classification
4. Evaluating performance with test accuracy and cross-validation
5. Visualizing results with confusion matrices and probability plots

For real-world applications:
- Use actual EEG data from OpenBCI or other devices
- Tune hyperparameters (filter bank, CSP components, classifier type)
- Implement real-time classification
- Add artifact removal and advanced preprocessing